# ETL arboviruses historical series

## 1. Load raw dataset and create row index

In [0]:
CATALOG = "workspace"
SCHEMA = "default"
TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"

RAW_TABLE = f"{TABLE_PREFIX}.raw_arboviroses"
OUTPUT_TABLE = f"{TABLE_PREFIX}.fato_arboviroses"

from pyspark.sql import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id, col

df_arb_raw = spark.table(RAW_TABLE)

# The source file contains three stacked tables. The row index is used only to locate each block.
w = Window.orderBy(monotonically_increasing_id())
df_arb_idx = df_arb_raw.withColumn("row_id", row_number().over(w))

display(df_arb_idx.select("row_id", "_c0").limit(80))

## 2. Define helper functions and month order

In [0]:
from functools import reduce
from pyspark.sql.functions import (
    col, lit, trim, upper, regexp_replace, when, create_map, element_at
)

VALID_MONTHS = [
    "JANEIRO", "FEVEREIRO", "MARÇO", "ABRIL", "MAIO", "JUNHO",
    "JULHO", "AGOSTO", "SETEMBRO", "OUTUBRO", "NOVEMBRO", "DEZEMBRO"
]

MONTH_ORDER = {
    "JANEIRO": 1,
    "FEVEREIRO": 2,
    "MARÇO": 3,
    "ABRIL": 4,
    "MAIO": 5,
    "JUNHO": 6,
    "JULHO": 7,
    "AGOSTO": 8,
    "SETEMBRO": 9,
    "OUTUBRO": 10,
    "NOVEMBRO": 11,
    "DEZEMBRO": 12,
}

month_order_map = create_map(*[x for item in MONTH_ORDER.items() for x in (lit(item[0]), lit(item[1]))])

def clean_int_col(column_name):
    # Removes thousand separators and other non-numeric characters before casting.
    cleaned = regexp_replace(trim(col(column_name).cast("string")), r"[^0-9-]", "")
    return when((cleaned == "") | cleaned.isNull(), None).otherwise(cleaned.cast("int"))

## 3. Define column maps for each disease

In [0]:
DENGUE_COLUMNS = {
    1998: {"casos": "_c1",  "obitos": None},
    1999: {"casos": "_c2",  "obitos": None},
    2000: {"casos": "_c3",  "obitos": None},
    2001: {"casos": "_c4",  "obitos": None},
    2002: {"casos": "_c5",  "obitos": None},
    2003: {"casos": "_c6",  "obitos": None},
    2004: {"casos": "_c7",  "obitos": None},
    2005: {"casos": "_c8",  "obitos": None},
    2006: {"casos": "_c9",  "obitos": None},
    2007: {"casos": "_c10", "obitos": None},
    2008: {"casos": "_c11", "obitos": None},
    2009: {"casos": "_c12", "obitos": "_c13"},
    2010: {"casos": "_c14", "obitos": "_c15"},
    2011: {"casos": "_c16", "obitos": "_c17"},
    2012: {"casos": "_c18", "obitos": "_c19"},
    2013: {"casos": "_c20", "obitos": "_c21"},
    2014: {"casos": "_c22", "obitos": "_c23"},
    2015: {"casos": "_c25", "obitos": "_c26"},
    2016: {"casos": "_c28", "obitos": "_c29"},
    2017: {"casos": "_c31", "obitos": "_c32"},
    2018: {"casos": "_c34", "obitos": "_c35"},
    2019: {"casos": "_c37", "obitos": "_c38"},
    2020: {"casos": "_c40", "obitos": "_c41"},
    2021: {"casos": "_c43", "obitos": "_c44"},
    2022: {"casos": "_c46", "obitos": "_c47"},
    2023: {"casos": "_c49", "obitos": "_c50"},
    2024: {"casos": "_c52", "obitos": "_c53"},
    2025: {"casos": "_c55", "obitos": "_c56"},
    2026: {"casos": "_c58", "obitos": "_c59"},
}

ZIKA_COLUMNS = {
    2016: {"casos": "_c1",  "obitos": "_c2"},
    2017: {"casos": "_c4",  "obitos": "_c5"},
    2018: {"casos": "_c7",  "obitos": "_c8"},
    2019: {"casos": "_c10", "obitos": "_c11"},
    2020: {"casos": "_c13", "obitos": "_c14"},
    2021: {"casos": "_c16", "obitos": "_c17"},
    2022: {"casos": "_c19", "obitos": "_c20"},
    2023: {"casos": "_c22", "obitos": "_c23"},
    2024: {"casos": "_c25", "obitos": "_c26"},
    2025: {"casos": "_c28", "obitos": "_c29"},
    2026: {"casos": "_c31", "obitos": "_c32"},
}

CHIKUNGUNYA_COLUMNS = ZIKA_COLUMNS.copy()

## 4. Create reusable transformation function

In [0]:
def transform_disease(df_base, row_ini, row_fim, disease_name, year_columns):
    df_filtered = (
        df_base
        .filter((col("row_id") >= row_ini) & (col("row_id") <= row_fim))
        .select(
            upper(trim(regexp_replace(col("_c0").cast("string"), "^\ufeff", ""))).alias("mes"),
            *[col(c) for c in df_base.columns if c not in ["_c0", "row_id"]]
        )
        .filter(col("mes").isin(VALID_MONTHS))
    )

    dfs = []

    for year, cols in year_columns.items():
        cases_col = cols["casos"]
        deaths_col = cols["obitos"]

        df_year = (
            df_filtered
            .select(
                col("mes"),
                lit(year).alias("ano"),
                clean_int_col(cases_col).alias("casos"),
                (clean_int_col(deaths_col) if deaths_col is not None else lit(0).cast("int")).alias("obitos"),
                lit(disease_name).alias("doenca")
            )
        )
        dfs.append(df_year)

    return (
        reduce(lambda a, b: a.unionByName(b), dfs)
        .filter(col("casos").isNotNull())
        .withColumn("obitos", when(col("obitos").isNull(), lit(0)).otherwise(col("obitos")))
        .withColumn("ordem_mes", element_at(month_order_map, col("mes")))
        .select("ano", "mes", "ordem_mes", "casos", "obitos", "doenca")
        .orderBy("doenca", "ano", "ordem_mes")
    )

## 5. Transform each disease

In [0]:
df_dengue = transform_disease(
    df_base=df_arb_idx,
    row_ini=5,
    row_fim=16,
    disease_name="dengue",
    year_columns=DENGUE_COLUMNS
)

df_zika = transform_disease(
    df_base=df_arb_idx,
    row_ini=26,
    row_fim=37,
    disease_name="zika",
    year_columns=ZIKA_COLUMNS
)

df_chikungunya = transform_disease(
    df_base=df_arb_idx,
    row_ini=45,
    row_fim=56,
    disease_name="chikungunya",
    year_columns=CHIKUNGUNYA_COLUMNS
)

display(df_dengue.orderBy("ano", "ordem_mes"))
display(df_zika.orderBy("ano", "ordem_mes"))
display(df_chikungunya.orderBy("ano", "ordem_mes"))

## 6. Union final arboviruses fact table

In [0]:
df_arboviroses_final = (
    df_dengue
    .unionByName(df_zika)
    .unionByName(df_chikungunya)
    .orderBy("doenca", "ano", "ordem_mes")
)

display(df_arboviroses_final)

## 7. Validate

In [0]:
print("Rows:", df_arboviroses_final.count())

display(
    df_arboviroses_final
    .groupBy("doenca", "ano")
    .count()
    .orderBy("doenca", "ano")
)

display(
    df_arboviroses_final
    .groupBy("doenca", "ano")
    .sum("casos", "obitos")
    .orderBy("doenca", "ano")
)

## 8. Save transformed table

In [0]:
df_arboviroses_final.write     .format("delta")     .mode("overwrite")     .option("overwriteSchema", "true")     .saveAsTable(OUTPUT_TABLE)